# Indian Startup Funding Analysis — Data Exploration

**Notebook 01 of 4** | Goal: Understand what raw data we have and what's broken in it.

## Context
We've sourced 4 public datasets covering Indian startup funding from 2015–2025. Each dataset has its own format, its own quirks, and its own idea of what "clean" data looks like.

This notebook is the first honest look at what we're working with — before any cleaning happens. The goal isn't to fix anything yet. The goal is to **see** what's broken so we know what to fix later.

## What We'll Do Here
1. Load each dataset raw, exactly as Kaggle gave it to us
2. Inspect column names, shapes, and data types
3. Spot early quality issues — missing values, weird formats, encoding problems
4. Write down everything we notice for the cleaning phase that follows

## Datasets in Scope
| File | Rows | Coverage |
|---|---|---|
| `startup_funding_raw_1.csv` | ~3,044 | 2015–2020, Indian |
| `startup_funding_raw_2.csv` | ~526 | Indian (no dates) |
| `startup_funding_raw_3.csv` | ~1,100 | 2020–2025, Indian |
| `startup_funding_raw_4.csv` | ~54,294 | Global — will filter for India |

In [3]:
# Core libraries for data manipulation
import pandas as pd
import numpy as np

# Display settings — make pandas show more columns and wider output
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

# So we don't get cluttered warnings during exploration
import warnings
warnings.filterwarnings('ignore')

print("Environment ready.")

Environment ready.


## Loading Dataset 1 — The Historical Anchor

**Source:** Kaggle — *Indian Startup Funding* by sudalairajkumar  
**Expected coverage:** January 2015 – September 2019/2020, ~3,044 rounds

We're going to load this dataset with zero pre-processing — exactly as the file came from Kaggle. The goal is to see what pandas encounters when it reads the raw CSV.

### What we'll inspect
- `.shape` — how many rows and columns
- `.head()` — the first 5 rows
- `.info()` — data types pandas inferred for each column
- `.isnull().sum()` — missing values per column
- `.duplicated().sum()` — fully duplicated rows

This is our reconnaissance. We don't fix anything — we just take notes.

In [4]:
# Load Dataset 1 — our historical anchor
df1 = pd.read_csv('../data/raw/startup_funding_raw_1.csv')

# Immediately check the shape — how big is this thing?
print(f"Dataset 1 shape: {df1.shape}")
print(f"→ {df1.shape[0]} rows, {df1.shape[1]} columns")

Dataset 1 shape: (3044, 10)
→ 3044 rows, 10 columns


In [5]:
# Look at the first 5 rows — what does the actual data look like?
df1.head()

,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


In [6]:
# Full inspection — data types + missing values
print("=" * 60)
print("DATASET 1 — COLUMN INFO")
print("=" * 60)
df1.info()

DATASET 1 — COLUMN INFO
<class 'pandas.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Sr No              3044 non-null   int64
 1   Date dd/mm/yyyy    3044 non-null   str  
 2   Startup Name       3044 non-null   str  
 3   Industry Vertical  2873 non-null   str  
 4   SubVertical        2108 non-null   str  
 5   City  Location     2864 non-null   str  
 6   Investors Name     3020 non-null   str  
 7   InvestmentnType    3040 non-null   str  
 8   Amount in USD      2084 non-null   str  
 9   Remarks            419 non-null    str  
dtypes: int64(1), str(9)
memory usage: 237.9 KB


In [7]:
# How many rows are exact duplicates of other rows?
duplicate_count = df1.duplicated().sum()
print(f"Exact duplicate rows in Dataset 1: {duplicate_count}")

# Also — how many unique companies?
unique_companies = df1['Startup Name'].nunique()
print(f"Unique company names (raw, before dedup): {unique_companies}")
print(f"→ That means {df1.shape[0] - unique_companies} rows share a name with another row")
print(f"  (Some of these are legit — same company raising multiple rounds — but some are duplicates)")

Exact duplicate rows in Dataset 1: 0
Unique company names (raw, before dedup): 2459
→ That means 585 rows share a name with another row
  (Some of these are legit — same company raising multiple rounds — but some are duplicates)


In [8]:
# Find company names that appear more than once
name_counts = df1['Startup Name'].value_counts()

# Show the top 15 most-repeated company names
print("Top 15 companies by number of rows in Dataset 1:")
print(name_counts.head(15))

print("\n" + "=" * 60)
print(f"Companies appearing more than once: {(name_counts > 1).sum()}")
print(f"Companies appearing exactly once: {(name_counts == 1).sum()}")

Top 15 companies by number of rows in Dataset 1:
Startup Name
Ola Cabs         8
Swiggy           8
Paytm            7
Meesho           6
NoBroker         6
Nykaa            6
Medinfi          6
UrbanClap        6
Uniphore         5
Grofers          5
Moglix           5
Toppr            5
Capital Float    5
Flipkart         5
Jugnoo           5
Name: count, dtype: int64

Companies appearing more than once: 393
Companies appearing exactly once: 2066


In [9]:
# Count missing values per column, sorted by worst first
missing = df1.isnull().sum().sort_values(ascending=False)
missing_pct = (df1.isnull().sum() / len(df1) * 100).round(1).sort_values(ascending=False)

# Combine both counts and percentages into one view
missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print("Missing data report — Dataset 1:")
print(missing_report)

Missing data report — Dataset 1:
                   Missing Count  Missing %
Remarks                     2625       86.2
Amount in USD                960       31.5
SubVertical                  936       30.7
City  Location               180        5.9
Industry Vertical            171        5.6
Investors Name                24        0.8
InvestmentnType                4        0.1
Sr No                          0        0.0
Date dd/mm/yyyy                0        0.0
Startup Name                   0        0.0


## Dataset 1 — Findings Summary

### Shape
- **3,044 rows × 10 columns**
- No exact duplicate rows
- 393 companies appear in multiple rounds (legitimate multi-round activity)

### Column Quality Issues
| Issue | Column(s) | Severity |
|---|---|---|
| Column names have spaces, typos, format info baked in | All 10 | High — rename all |
| `Remarks` is 86% empty | `Remarks` | High — drop column |
| `Amount in USD` is 31.5% missing | `Amount in USD` | Medium — decide strategy (keep for count, exclude from amount analysis) |
| `SubVertical` is 30.7% missing | `SubVertical` | Medium — may drop or merge with Industry Vertical |
| Every column read as `str`, including amount and date | Amount, Date | Critical — need type conversion |

### Content Quality Issues
| Issue | Example | Fix |
|---|---|---|
| Amounts in Indian comma format (lakhs/crores) | `20,00,00,000` | Strip commas, convert to integer |
| Dates as strings in DD/MM/YYYY | `09/01/2020` | Parse with `pd.to_datetime(..., format='%d/%m/%Y')` |
| Unicode apostrophes in names | `BYJU'S` (curly `'`) | Normalize Unicode |
| URLs in name fields | `https://www.wealthbucket.in/` | Detect and clean |
| Inconsistent stage labels | `Pre-series A` vs `Series A` | Build stage taxonomy |

### Decisions Carried Forward
- ✓ Will **drop** `Remarks` column in cleaning
- ✓ Will **rename** all columns to clean `snake_case`
- ✓ Will **keep** rows with missing amounts (they're still valid rounds by count)
- ⚠ Will investigate: is `SubVertical` useful or drop it too?

---

## Loading Dataset 2 — The Thin Supplement

**Source:** Kaggle — *Indian Startups Funding Data* by omkargowda  
**Expected coverage:** ~526 rounds, **no date column**

This dataset is small and has no dates. That's a serious limitation for a time-series analysis.

### What we're watching for
- Can we salvage it? Does it have value beyond what Dataset 1 already gives us?
- Are there companies or metadata here that don't exist in Dataset 1?
- What's in the `About Company` column — is it useful context?

**Preliminary call:** If this dataset duplicates Dataset 1's content without adding anything new, we drop it. If it has unique companies or useful metadata, we salvage what we can.

In [10]:
# Load Dataset 2 — the thin supplement (no dates)
df2 = pd.read_csv('../data/raw/startup_funding_raw_2.csv')

# Shape first
print(f"Dataset 2 shape: {df2.shape}")
print(f"→ {df2.shape[0]} rows, {df2.shape[1]} columns\n")

# First 5 rows
print("First 5 rows:")
df2.head()

Dataset 2 shape: (526, 6)
→ 526 rows, 6 columns

First 5 rows:


,Company Name,Industry,Round/Series,Amount,Location,About Company
0,TheCollegeFever,"Brand Marketing, Event Promotion, Marketing, Sponsorship, Ticketing",Seed,250000,"Bangalore, Karnataka, India","TheCollegeFever is a hub for fun, fiesta and frolic of Colleges."
1,Happy Cow Dairy,"Agriculture, Farming",Seed,"₹40,000,000","Mumbai, Maharashtra, India",A startup which aggregates milk from dairy farmers in rural Maharashtra.
2,MyLoanCare,"Credit, Financial Services, Lending, Marketplace",Series A,"₹65,000,000","Gurgaon, Haryana, India",Leading Online Loans Marketplace in India
3,PayMe India,"Financial Services, FinTech",Angel,2000000,"Noida, Uttar Pradesh, India",PayMe India is an innovative FinTech organization which offers short term fi...
4,Eunimart,"E-Commerce Platforms, Retail, SaaS",Seed,—,"Hyderabad, Andhra Pradesh, India",Eunimart is a one stop solution for merchants to create a difference by sell...


In [11]:
# Missing values breakdown
print("Dataset 2 — Missing values:")
print(df2.isnull().sum())
print(f"\nExact duplicate rows: {df2.duplicated().sum()}")
print(f"Unique companies: {df2['Company Name'].nunique()} out of {len(df2)} rows")

# Check if em-dash is hiding in Amount column
print(f"\nSample of 'Amount' values containing '—' (em-dash):")
em_dash_count = df2['Amount'].astype(str).str.contains('—', na=False).sum()
print(f"Rows with em-dash in Amount: {em_dash_count}")

Dataset 2 — Missing values:
Company Name     0
Industry         0
Round/Series     0
Amount           0
Location         0
About Company    0
dtype: int64

Exact duplicate rows: 1
Unique companies: 525 out of 526 rows

Sample of 'Amount' values containing '—' (em-dash):
Rows with em-dash in Amount: 148


In [14]:
# Sample rows where Amount is em-dash
print("=" * 60)
print("Sample rows where Amount is '—' (em-dash)")
print("=" * 60)
em_dash_rows = df2[df2['Amount'].astype(str).str.contains('—', na=False)]
print(em_dash_rows[['Company Name', 'Round/Series', 'Amount']].head(5))

# Sample rows where Amount has INR symbol
print("\n" + "=" * 60)
print("Sample rows where Amount has ₹ (INR symbol)")
print("=" * 60)
inr_rows = df2[df2['Amount'].astype(str).str.contains('₹', na=False)]
print(f"Total rows with ₹: {len(inr_rows)}")
print(inr_rows[['Company Name', 'Round/Series', 'Amount']].head(5))

# Pre-calculate all three counts (avoid backslash in f-string)
em_dash_count = df2['Amount'].astype(str).str.contains('—', na=False).sum()
inr_count = df2['Amount'].astype(str).str.contains('₹', na=False).sum()
plain_pattern = r'^\d+$'
plain_count = df2['Amount'].astype(str).str.match(plain_pattern).sum()

# Print summary
print("\n" + "=" * 60)
print("Breakdown of Amount column content types")
print("=" * 60)
print(f"Em-dash rows:      {em_dash_count}")
print(f"INR symbol rows:   {inr_count}")
print(f"Plain number rows: {plain_count}")
print(f"Other/unknown:     {len(df2) - em_dash_count - inr_count - plain_count}")

Sample rows where Amount is '—' (em-dash)
                  Company Name Round/Series Amount
4                     Eunimart         Seed      —
8                 Freightwalla         Seed      —
9           Microchip Payments         Seed      —
10  BizCrum Infotech Pvt. Ltd.         Seed      —
11                     Emojifi         Seed      —

Sample rows where Amount has ₹ (INR symbol)
Total rows with ₹: 144
       Company Name Round/Series        Amount
1   Happy Cow Dairy         Seed   ₹40,000,000
2        MyLoanCare     Series A   ₹65,000,000
6         Tripshelf         Seed   ₹16,000,000
7      Hyperdata.IO        Angel   ₹50,000,000
15          Pitstop         Seed  ₹100,000,000

Breakdown of Amount column content types
Em-dash rows:      148
INR symbol rows:   144
Plain number rows: 175
Other/unknown:     59


In [15]:
# Find rows that aren't em-dash, INR symbol, OR plain digit
mask_em = df2['Amount'].astype(str).str.contains('—', na=False)
mask_inr = df2['Amount'].astype(str).str.contains('₹', na=False)
mask_plain = df2['Amount'].astype(str).str.match(r'^\d+$')

# The "other" rows are what's left
mask_other = ~mask_em & ~mask_inr & ~mask_plain

mystery_rows = df2[mask_other]
print(f"Mystery rows: {len(mystery_rows)}")
print("\nSample of unknown amount formats:")
print(mystery_rows[['Company Name', 'Round/Series', 'Amount']].head(15))

# Show unique values (up to 20) of these weird amounts
print("\n" + "=" * 60)
print("Unique values in mystery amount rows (first 20):")
print("=" * 60)
print(mystery_rows['Amount'].unique()[:20])

Mystery rows: 59

Sample of unknown amount formats:
                                Company Name              Round/Series          Amount
86                                       WHR                      Seed        $143,145
90                                  SBI Life            Private Equity    $742,000,000
93          NoPaperForms Solutions Pvt. Ltd.                  Series B      $3,980,000
95                                AuthMetrik                     Grant         $10,000
101                                   Swiggy                  Series H  $1,000,000,000
102                               Milkbasket                  Series A      $7,000,000
104                                    Toppr                  Series C     $35,000,000
106                          Vivriti Capital  Venture - Series Unknown     $28,500,000
108                              Impact Guru                  Series A      $2,000,000
114                                OneAssist            Debt Financing      $2

## Dataset 2 — Findings Summary

### Shape
- **526 rows × 6 columns**
- 1 exact duplicate row
- 525 unique companies (essentially 1 round per company — not longitudinal)

### Structural Issues
| Issue | Severity |
|---|---|
| **No date column** | 🚨 Critical — blocks any time-series use |
| Only 6 columns (missing investor, sub-vertical) | Medium |
| Industry column is multi-tag comma-separated | Medium |
| Location includes state + country | Minor |

### Amount Column — Four Formats Detected
| Format | Count | % | Currency |
|---|---|---|---|
| Em-dash `—` (hidden missing) | 148 | 28.1% | N/A |
| INR symbol `₹` + Western commas | 144 | 27.4% | INR |
| Plain integers | 175 | 33.3% | **Ambiguous** |
| Dollar symbol `$` | 59 | 11.2% | USD |

**Critical insight:** Pandas counted `0` missing values because the em-dash isn't recognized as null. Visual inspection caught this.

### Strategic Decision — Relegate to Enrichment Source
This dataset cannot support primary analysis due to:
- Missing dates — blocks any time-series, cohort, or trend analysis
- Missing investor data
- Ambiguous currency on 33% of amount rows
- Largely overlaps with Dataset 1 (no unique temporal coverage)

**Plan:** Use Dataset 2 only to enrich company metadata (About Company text, location state detail, multi-tag industry info). It will **not** contribute rows to the final combined funding dataset.

This is a deliberate analytical judgment call, not a data failure. **It will be documented in the methodology log.**

---

## Loading Dataset 3 — The Recent Era

**Source:** Kaggle — *Indian Startup Funding Dataset (2020–2025)* by vagdevititikshag  
**Expected coverage:** 2020–2025, ~1,100 rounds

This dataset covers the years Dataset 1 misses — the ZIRP boom (2020–2021), the funding winter (2022–2023), and the AI-driven recovery (2024–2025).

### What we're watching for
- **Date format:** Spot-check showed `DD-MM-YYYY` (different from Dataset 1's `DD/MM/YYYY`)
- **Amount column:** Appears to be clean USD integers — almost suspicious after Dataset 2
- **Data quality:** Spot-check showed `Housejoy` labeled as `EdTech/K12` which is wrong — Housejoy is a home services company. Need to check for systemic labeling errors.
- **Column names:** More consistent than Dataset 1 (`InvestmentAmount_USD` vs `Amount in USD`)

**Preliminary hypothesis:** This dataset looks cleaner on the surface but may have factual/categorization errors that plain structural checks won't catch. We need to be skeptical.

In [16]:
# Load Dataset 3 — the recent era (2020-2025)
df3 = pd.read_csv('../data/raw/startup_funding_raw_3.csv')

# Shape first
print(f"Dataset 3 shape: {df3.shape}")
print(f"→ {df3.shape[0]} rows, {df3.shape[1]} columns\n")

# First 5 rows
print("First 5 rows:")
df3.head()

Dataset 3 shape: (1100, 8)
→ 1100 rows, 8 columns

First 5 rows:


,Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date
0,Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed,199000,19-04-2023
1,Groww,Media,Streaming,Bengaluru,IFC,Seed,1668000,28-01-2025
2,Groww,Mobility,Ride Sharing,Hyderabad,"Nexus Venture Partners, Peak XV",Series B,38052000,14-03-2021
3,FarmBox,Consumer Electronics,Wearables,Gurugram,"Kalaari Capital, Y Combinator",Seed,455000,11-09-2023
4,Udaan,RealEstate,Rental Tech,Mumbai,Bessemer Venture Partners,Seed,89000,31-01-2024


In [17]:
# Full inspection
print("=" * 60)
print("DATASET 3 — COLUMN INFO")
print("=" * 60)
df3.info()

print("\n" + "=" * 60)
print("Missing values report:")
print("=" * 60)
missing3 = pd.DataFrame({
    'Missing Count': df3.isnull().sum(),
    'Missing %': (df3.isnull().sum() / len(df3) * 100).round(1)
}).sort_values('Missing Count', ascending=False)
print(missing3)

print("\n" + "=" * 60)
print(f"Exact duplicate rows: {df3.duplicated().sum()}")
print(f"Unique companies: {df3['Startup'].nunique()} out of {len(df3)} rows")
print("=" * 60)

DATASET 3 — COLUMN INFO
<class 'pandas.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Startup               1100 non-null   str  
 1   Industry              1100 non-null   str  
 2   SubVertical           1100 non-null   str  
 3   City                  1100 non-null   str  
 4   Investors             1100 non-null   str  
 5   InvestmentType        1100 non-null   str  
 6   InvestmentAmount_USD  1100 non-null   int64
 7   Date                  1100 non-null   str  
dtypes: int64(1), str(7)
memory usage: 68.9 KB

Missing values report:
                      Missing Count  Missing %
Startup                           0        0.0
Industry                          0        0.0
SubVertical                       0        0.0
City                              0        0.0
Investors                         0        0.0
InvestmentType                    0   

In [18]:
# How many times do companies repeat in Dataset 3?
name_counts3 = df3['Startup'].value_counts()

print("Top 20 most repeated companies in Dataset 3:")
print(name_counts3.head(20))

print("\n" + "=" * 60)
print(f"Companies appearing more than 5 times:  {(name_counts3 > 5).sum()}")
print(f"Companies appearing more than 10 times: {(name_counts3 > 10).sum()}")
print(f"Companies appearing more than 20 times: {(name_counts3 > 20).sum()}")
print("=" * 60)

# Look at all rows for one heavily repeated company
print("\nAll rows for 'Groww' in Dataset 3:")
print(df3[df3['Startup'] == 'Groww'][['Startup', 'Industry', 'SubVertical', 
                                       'InvestmentType', 'InvestmentAmount_USD', 
                                       'Date']].to_string())

Top 20 most repeated companies in Dataset 3:
Startup
DriveEdge           15
Ola                 14
LogiMart            14
FreshTech           14
Oyo                 14
Udaan               13
Porter              13
Delhivery           13
Housejoy            12
FarmBox             12
SkillFlow           12
CoreConnect         12
HomeAnalytics       12
NextSense           11
QuantumSolutions    11
HyperLabs           11
Bounce              11
AgriHive            11
LogiSense           11
NeoLabs             11
Name: count, dtype: int64

Companies appearing more than 5 times:  109
Companies appearing more than 10 times: 25
Companies appearing more than 20 times: 0

All rows for 'Groww' in Dataset 3:
    Startup    Industry   SubVertical InvestmentType  InvestmentAmount_USD        Date
1     Groww       Media     Streaming           Seed               1668000  28-01-2025
2     Groww    Mobility  Ride Sharing       Series B              38052000  14-03-2021
13    Groww     FinTech     InsurT

## Dataset 3 — Findings Summary

### Shape
- **1,100 rows × 8 columns**
- 0 exact duplicate rows
- Only **180 unique companies** — averaging 6.1 rows per company (suspicious)

### 🚨 Critical Finding — Dataset Contains Synthetic/Fabricated Data

Investigation revealed that this dataset is largely AI-generated, not real funding data:

**Evidence:**
1. Known companies (Groww, Housejoy, Udaan) appear 6–14 times each with completely wrong and inconsistent industry classifications
2. Groww (a stock investment app) is labeled as: Media, Mobility, SaaS, Retail, Enterprise Security — none correct
3. Unrecognizable company names ("DriveEdge", "LogiMart", "FreshTech", "SkillFlow") that don't exist in the Indian startup ecosystem
4. Phantom companies each appear 11–15 times with varying fake attributes

**Why this matters:**
- Pandas structural checks showed "0 missing values" and "0 duplicates" — appeared clean
- Only manual inspection + domain knowledge caught the fabrication
- Data validation must include factual verification, not just structural checks

### Strategic Decision — Drop As Primary Source
Dataset 3 excluded from final combined dataset. Will be documented in methodology log.

---

## Loading Dataset 4 — The Crunchbase Beast

**Source:** Kaggle — *StartUp Investments (Crunchbase)* by arindam235  
**Coverage:** ~54,294 rows, global — must filter for India  
**Size:** 12.5 MB — largest dataset by far

This is structurally the most different dataset. Key things to know before loading:

- One row per **company** (not per round)
- Funding amounts split across 20+ round-type columns (`round_A`, `round_B`, `seed`, `venture` etc.)
- Global data — we filter by `country_code = 'IND'` to get Indian companies
- Requires **reshaping from wide to long format** using `pd.melt()`

### Why This Dataset Matters
It provides company-level metadata (founding year, status, total funding) that the other datasets lack. Even though it's not round-level data natively, after reshaping it gives us additional Indian companies to enrich our primary dataset.

In [20]:
# Load Dataset 4 — Crunchbase global dataset
# encoding='latin-1' because file contains non-UTF-8 characters (Windows-era export)
df4 = pd.read_csv('../data/raw/startup_funding_raw_4.csv', 
                  encoding='latin-1',
                  low_memory=False)

# Shape first
print(f"Dataset 4 shape: {df4.shape}")
print(f"→ {df4.shape[0]} rows, {df4.shape[1]} columns\n")

# Show all column names
print("All columns:")
for i, col in enumerate(df4.columns):
    print(f"  {i:02d}. {col}")

Dataset 4 shape: (54294, 39)
→ 54294 rows, 39 columns

All columns:
  00. permalink
  01. name
  02. homepage_url
  03. category_list
  04.  market 
  05.  funding_total_usd 
  06. status
  07. country_code
  08. state_code
  09. region
  10. city
  11. funding_rounds
  12. founded_at
  13. founded_month
  14. founded_quarter
  15. founded_year
  16. first_funding_at
  17. last_funding_at
  18. seed
  19. venture
  20. equity_crowdfunding
  21. undisclosed
  22. convertible_note
  23. debt_financing
  24. angel
  25. grant
  26. private_equity
  27. post_ipo_equity
  28. post_ipo_debt
  29. secondary_market
  30. product_crowdfunding
  31. round_A
  32. round_B
  33. round_C
  34. round_D
  35. round_E
  36. round_F
  37. round_G
  38. round_H


In [22]:
# Filter for India only
df4_india = df4[df4['country_code'] == 'IND'].copy()

print(f"Global rows:        {len(df4)}")
print(f"Indian rows only:   {len(df4_india)}")
print(f"Filtered out:       {len(df4) - len(df4_india)} non-Indian rows")
print(f"India = {(len(df4_india)/len(df4)*100):.1f}% of the global dataset")

# Strip whitespace from ALL column names immediately
df4_india.columns = df4_india.columns.str.strip()

# Verify the fix
print("\nColumn names after stripping whitespace:")
print(list(df4_india.columns))

# Now peek at Indian companies
print("\nFirst 5 Indian companies:")
df4_india[['name', 'category_list', 'city', 'founded_year',
           'funding_rounds', 'funding_total_usd']].head()

Global rows:        54294
Indian rows only:   849
Filtered out:       53445 non-Indian rows
India = 1.6% of the global dataset

Column names after stripping whitespace:
['permalink', 'name', 'homepage_url', 'category_list', 'market', 'funding_total_usd', 'status', 'country_code', 'state_code', 'region', 'city', 'funding_rounds', 'founded_at', 'founded_month', 'founded_quarter', 'founded_year', 'first_funding_at', 'last_funding_at', 'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H']

First 5 Indian companies:


,name,category_list,city,founded_year,funding_rounds,funding_total_usd
70,1CLICK,|Chat|Mobile|,Bangalore,2012.0,1.0,-
103,21Diamonds,|E-Commerce|,Gurgaon,2012.0,1.0,"63,69,507"
126,24x7 Learning,|Systems|Education|,Bangalore,2001.0,1.0,"40,00,000"
226,3DSoC,|3D|Mobile|,Bangalore,2006.0,2.0,"20,65,000"
432,91Mobiles,|Mobile|,Gurgaon,2010.0,1.0,"10,00,000"


In [23]:
# Full inspection of Indian slice
print("=" * 60)
print("DATASET 4 (India only) — COLUMN INFO")
print("=" * 60)
df4_india.info()

print("\n" + "=" * 60)
print("Missing values report (top 15 worst columns):")
print("=" * 60)
missing4 = pd.DataFrame({
    'Missing Count': df4_india.isnull().sum(),
    'Missing %': (df4_india.isnull().sum() / len(df4_india) * 100).round(1)
}).sort_values('Missing Count', ascending=False)
print(missing4.head(15))

print("\n" + "=" * 60)
print(f"Exact duplicate rows: {df4_india.duplicated().sum()}")
print(f"Unique companies: {df4_india['name'].nunique()}")
print("=" * 60)

DATASET 4 (India only) — COLUMN INFO
<class 'pandas.DataFrame'>
Index: 849 entries, 70 to 49396
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   permalink             849 non-null    str    
 1   name                  849 non-null    str    
 2   homepage_url          834 non-null    str    
 3   category_list         816 non-null    str    
 4   market                816 non-null    str    
 5   funding_total_usd     849 non-null    str    
 6   status                832 non-null    str    
 7   country_code          849 non-null    str    
 8   state_code            0 non-null      str    
 9   region                849 non-null    str    
 10  city                  844 non-null    str    
 11  funding_rounds        849 non-null    float64
 12  founded_at            683 non-null    str    
 13  founded_month         682 non-null    str    
 14  founded_quarter       682 non-null    str    
 15 

In [24]:
# Check the funding round columns specifically
round_cols = ['seed', 'venture', 'angel', 'grant', 
              'private_equity', 'round_A', 'round_B', 
              'round_C', 'round_D', 'round_E', 'round_F']

print("Funding round columns — how many Indian companies have non-zero values:")
print("=" * 55)

for col in round_cols:
    if col in df4_india.columns:
        # Count non-zero, non-null values
        non_zero = (pd.to_numeric(df4_india[col], errors='coerce') > 0).sum()
        pct = (non_zero / len(df4_india) * 100)
        print(f"  {col:<20} {non_zero:>4} companies  ({pct:.1f}%)")

Funding round columns — how many Indian companies have non-zero values:
  seed                  202 companies  (23.8%)
  venture               357 companies  (42.0%)
  angel                  65 companies  (7.7%)
  grant                   3 companies  (0.4%)
  private_equity         37 companies  (4.4%)
  round_A               133 companies  (15.7%)
  round_B                92 companies  (10.8%)
  round_C                37 companies  (4.4%)
  round_D                16 companies  (1.9%)
  round_E                 9 companies  (1.1%)
  round_F                 3 companies  (0.4%)


## Dataset 4 — Findings Summary (India Slice)

### Shape After Filtering
- **54,294 global → 849 Indian companies** (1.6% of global dataset)
- 0 duplicate rows
- 849 unique companies — one row per company (company-level, not round-level)

### Column Issues
| Issue | Column | Fix |
|---|---|---|
| `state_code` entirely empty (0 non-null) | `state_code` | Drop column |
| `funding_total_usd` stored as string | `funding_total_usd` | Strip commas, convert to numeric |
| Column names had leading/trailing spaces | All columns | Already fixed with `.str.strip()` |
| Em-dash `—` hiding as missing in amounts | `funding_total_usd` | Replace with NaN |
| `category_list` uses pipe separators | `category_list` | Extract primary category |
| `founded_year` is float64, not int | `founded_year` | Convert to Int64 |

### Round Column Distribution (Indian companies)
- `venture`: 357 (42%) — Crunchbase catch-all for unspecified series
- `seed`: 202 (24%)
- `round_A`: 133 (16%)
- `round_B`: 92 (11%)
- Healthy funnel shape — decreasing counts at each stage ✓

### Strategic Decision — Use As Secondary Primary Source
Dataset 4 provides:
- 849 unique Indian companies with founding year + total funding
- Round-type amounts (needs reshaping from wide to long)
- Status information (operating/acquired/closed) — unique to this dataset
- Will be merged with Dataset 1 on company name after cleaning

### Reshaping Required
Wide format (one row per company, rounds as columns) must be converted to long format (one row per round) using `pd.melt()` in the cleaning phase.

---

# Exploration Complete — What We Found

## Dataset Inventory

| Dataset | Rows | Status | Role in Project |
|---|---|---|---|
| Dataset 1 | 3,044 | ✓ Primary | Round-level data, 2015–2020 |
| Dataset 2 | 526 | Enrichment only | No dates, currency chaos |
| Dataset 3 | 1,100 | 🚨 Dropped | Synthetic/fabricated data |
| Dataset 4 | 849 (India) | ✓ Secondary primary | Company metadata + round amounts |

**Usable rows going into cleaning: ~3,893**

## Key Problems We'll Solve in Notebook 02

| Problem | Datasets Affected | Technique |
|---|---|---|
| Column name inconsistencies | All | Rename to snake_case |
| Indian comma format amounts | 1, 4 | Strip commas, convert to numeric |
| Mixed currency (USD/INR/₹/$) | 2 | Currency detection + conversion |
| Hidden missing values (em-dash) | 2, 4 | Replace with NaN |
| Date format variations (/ vs -) | 1, 3 | Source-specific format parsing |
| Unicode apostrophes in names | 1 | Unicode normalization |
| URLs in company name fields | 1 | Regex detection + cleaning |
| Inconsistent stage labels | 1, 2 | Stage taxonomy mapping |
| Pipe-separated categories | 4 | Primary category extraction |
| Wide → long reshaping | 4 | pd.melt() |
| Company name deduplication | 1 + 4 | rapidfuzz fuzzy matching |
| Fabricated data detection | 3 | Domain knowledge + repeat analysis |

## Most Important Analytical Lesson From This Notebook

> Structural cleanliness (zero nulls, zero duplicates) does not mean data is valid.
> Dataset 3 passed every automated check but was entirely fabricated.
> Dataset 2 showed zero missing values but had 148 em-dashes hiding real nulls.
> A data analyst must combine automated checks with domain knowledge and manual inspection.

## Next: Notebook 02 — The Cleaning Pipeline